<a href="https://colab.research.google.com/github/mahmudscode/Vehicle-Object-Detection-Dataset-5-Classes-/blob/main/vehicle_object_detection_(1)_(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("hammadjavaid/vehicle-object-detection-dataset-5-classes")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'vehicle-object-detection-dataset-5-classes' dataset.
Path to dataset files: /kaggle/input/vehicle-object-detection-dataset-5-classes


In [ ]:
import os

# Use the path obtained from kagglehub.dataset_download in the previous cell
dataset_path = path

# Check if the directory exists
if os.path.exists(dataset_path):
    print(f"Contents of '{dataset_path}':")
    for item in os.listdir(dataset_path):
        print(item)
else:
    print(f"Error: Directory '{dataset_path}' not found.")

Contents of '/kaggle/input/vehicle-object-detection-dataset-5-classes':
vehicles


In [ ]:
import os

# First, let's check what's actually in the dataset path
print(f"Exploring dataset structure in: {dataset_path}\n")

# Function to explore directory structure
def explore_directory(path, max_depth=5, current_depth=0):
    if current_depth > max_depth:
        return

    indent = "  " * current_depth # Initialize indent here
    try:
        items = os.listdir(path)
        print(f"{indent}Contents of '{path}':")

        for item in items:
            item_path = os.path.join(path, item)
            if os.path.isdir(item_path):
                print(f"{indent}  📁 {item}/")
                if current_depth < max_depth:
                    explore_directory(item_path, max_depth, current_depth + 1)
            else:
                print(f"{indent}  📄 {item}")
    except Exception as e:
        print(f"{indent}Error accessing {path}: {e}")

explore_directory(dataset_path, max_depth=2)

# Now try to find the correct dataset subdirectory
possible_paths = [
    os.path.join(dataset_path, 'dataset'),
    dataset_path,  # Sometimes the dataset folder IS the root
]

dataset_sub_path = None
for path in possible_paths:
    if os.path.exists(path):
        # Check if this path has 'images' and 'labels' folders
        contents = os.listdir(path)
        if 'images' in contents and 'labels' in contents:
            dataset_sub_path = path
            print(f"\n✅ Found dataset directory: {dataset_sub_path}")
            break

if dataset_sub_path is None:
    # If still not found, search for images and labels folders
    for root, dirs, files in os.walk(dataset_path):
        if 'images' in dirs and 'labels' in dirs:
            dataset_sub_path = root
            print(f"\n✅ Found dataset directory: {dataset_sub_path}")
            break

if dataset_sub_path:
    print(f"\nContents of dataset directory:")
    for item in os.listdir(dataset_sub_path):
        print(f"  - {item}")
else:
    print("\n❌ Could not find dataset directory with 'images' and 'labels' folders")

Exploring dataset structure in: /kaggle/input/vehicle-object-detection-dataset-5-classes

Contents of '/kaggle/input/vehicle-object-detection-dataset-5-classes':
  📁 vehicles/
  Contents of '/kaggle/input/vehicle-object-detection-dataset-5-classes/vehicles':
    📄 data.yaml
    📁 val/
    Contents of '/kaggle/input/vehicle-object-detection-dataset-5-classes/vehicles/val':
      📁 labels/
      📁 images/
    📁 test/
    Contents of '/kaggle/input/vehicle-object-detection-dataset-5-classes/vehicles/test':
      📁 labels/
      📁 images/
    📁 train/
    Contents of '/kaggle/input/vehicle-object-detection-dataset-5-classes/vehicles/train':
      📁 labels/
      📁 images/

✅ Found dataset directory: /kaggle/input/vehicle-object-detection-dataset-5-classes/vehicles/val

Contents of dataset directory:
  - labels
  - images


In [ ]:
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 64.5 MB/s eta 0:00:00


In [ ]:
import ultralytics

print(f"Ultralytics version: {ultralytics.__version__}")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics version: 8.4.37


In [ ]:
import yaml
from pathlib import Path

# Dataset root (read-only)
# Use the 'path' variable from kagglehub.dataset_download and append 'vehicles'
DATASET_DIR = Path(path) / "vehicles"

# Writable directory
WORKING_DIR = Path("/kaggle/working")

# ---- Read existing data.yaml ----
original_yaml = DATASET_DIR / "data.yaml"

if not original_yaml.exists():
    raise FileNotFoundError(f"❌ data.yaml not found in dataset directory: {original_yaml}")

with open(original_yaml, "r") as f:
    original_data = yaml.safe_load(f)

# Extract class info
classes = original_data.get("names", [])
nc = original_data.get("nc", len(classes))

if not classes or nc == 0:
    raise ValueError("❌ No classes found in original data.yaml")

# ---- Create new YOLO config ----
yolo_data_config = {
    "path": str(DATASET_DIR.resolve()),
    "train": "train/images",
    "val": "val/images",
    "test": "test/images",
    "nc": nc,
    "names": classes
}

# Save new YAML
yolo_yaml_path = WORKING_DIR / "data_yolo.yaml"
# Ensure the parent directory exists before writing the file
yolo_yaml_path.parent.mkdir(parents=True, exist_ok=True)
with open(yolo_yaml_path, "w") as f:
    yaml.dump(yolo_data_config, f, sort_keys=False)

# Print confirmation
print(f"✅ data_yolo.yaml file saved to: {yolo_yaml_path}\n")
print("Content of data_yolo.yaml:")
print(yolo_yaml_path.read_text())

✅ data_yolo.yaml file saved to: /kaggle/working/data_yolo.yaml

Content of data_yolo.yaml:
path: /kaggle/input/vehicle-object-detection-dataset-5-classes/vehicles
train: train/images
val: val/images
test: test/images
nc: 5
names:
- bus
- car
- pickup
- truck
- van



In [ ]:
from ultralytics import YOLO
import torch
import gc
import os
import shutil

# ─────────────────────────────────────
# CONFIG — Session 1 of 2 (epochs 1 → 25)
# ─────────────────────────────────────
MODEL_SIZE     = 's'
EPOCHS         = 25            # Session 1: ep 1-25
IMG_SIZE       = 1024
BATCH          = 32
PROJECT        = '/content/runs'
RUN_NAME       = 'yolov8s_a100'
DRIVE_SAVE_DIR = '/content/drive/MyDrive/YOLOv8_weights'

# ─────────────────────────────────────
# GPU
# ─────────────────────────────────────
torch.cuda.empty_cache()
gc.collect()

print(f"🚀 GPU: {torch.cuda.get_device_name(0)}")
print(f"   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# ─────────────────────────────────────
# Load Model — fresh start
# ─────────────────────────────────────
model = YOLO(f'yolov8{MODEL_SIZE}.pt')
print(f"✅ Fresh start: YOLOv8{MODEL_SIZE}")
print(f"   Parameters: {sum(p.numel() for p in model.model.parameters()):,}")

# ─────────────────────────────────────
# Train — Session 1
# ─────────────────────────────────────
print(f"\n⚗️ Session 1: epoch 1 → {EPOCHS} | imgsz={IMG_SIZE} | batch={BATCH}\n")

results = model.train(
    data=str(yolo_yaml_path), # Corrected path to data.yaml
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH,
    device=0,
    cache='disk',
    workers=4,
    resume=False,
    patience=10,
    save=True,
    save_period=5,
    project=PROJECT,
    name=RUN_NAME,
    exist_ok=True,
    pretrained=True,
    optimizer='AdamW',
    verbose=True,
    seed=42,
    deterministic=False,

    lr0=0.01,
    lrf=0.01,
    momentum=0.937,
    weight_decay=0.0005,
    warmup_epochs=3.0,
    warmup_momentum=0.8,

    cos_lr=True,
    close_mosaic=5,            # disable mosaic at epoch 20 (last 5 of 25)

    box=9.0,
    cls=0.5,
    dfl=1.5,

    amp=True,
    fraction=1.0,
    rect=False,
    single_cls=False,
    profile=False,

    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    degrees=5.0,
    translate=0.1,
    scale=0.6,
    shear=2.0,
    perspective=0.0001,
    flipud=0.0,
    fliplr=0.5,
    mosaic=1.0,
    mixup=0.1,
    copy_paste=0.1,
)

print("\n✅ Session 1 complete!")

# ─────────────────────────────────────
# Save to Google Drive immediately
# ─────────────────────────────────────
os.makedirs(DRIVE_SAVE_DIR, exist_ok=True)

best_src = f'{PROJECT}/{RUN_NAME}/weights/best.pt'
last_src = f'{PROJECT}/{RUN_NAME}/weights/last.pt'

shutil.copy(best_src, f'{DRIVE_SAVE_DIR}/best_ep{EPOCHS}.pt')
shutil.copy(last_src, f'{DRIVE_SAVE_DIR}/last_ep{EPOCHS}.pt')

# Also save as generic name for easy resume
shutil.copy(last_src, f'{DRIVE_SAVE_DIR}/last.pt')

print(f"\n💾 Saved to Google Drive: {DRIVE_SAVE_DIR}")
print(f"   → best_ep{EPOCHS}.pt")
print(f"   → last_ep{EPOCHS}.pt")
print(f"   → last.pt  (use this for session 2 resume)")

# ─────────────────────────────────────
# Quick Validation
# ─────────────────────────────────────
del model
torch.cuda.empty_cache()
gc.collect()

best_model = YOLO(best_src)

val_results = best_model.val(data=str(yolo_yaml_path), imgsz=IMG_SIZE, device=0) # Corrected path to data.yaml
print(f"\n📊 Standard:  mAP50={val_results.box.map50:.4f}  mAP50-95={val_results.box.map:.4f}")

tta_results = best_model.val(data=str(yolo_yaml_path), imgsz=IMG_SIZE, device=0, augment=True) # Corrected path to data.yaml
print(f"📊 TTA:       mAP50={tta_results.box.map50:.4f}  mAP50-95={tta_results.box.map:.4f}")

kaggle_best = 0.8968
gain = tta_results.box.map50 - kaggle_best
print(f"\n🎯 Gain over Kaggle baseline (0.8968): {gain:+.4f}")
print(f"   Session 2 will continue epoch 26 → 50")
print(f"   last.pt is ready in Drive for resume ✅")

🚀 GPU: NVIDIA A100-SXM4-40GB
   VRAM: 42.4 GB
✅ Fresh start: YOLOv8s
   Parameters: 11,166,560

⚗️ Session 1: epoch 1 → 25 | imgsz=1024 | batch=32

Ultralytics 8.4.37 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (NVIDIA A100-SXM4-40GB, 40441MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=9.0, cache=disk, cfg=None, classes=None, close_mosaic=5, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.1, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/kaggle/working/data_yolo.yaml, degrees=5.0, deterministic=False, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=25, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1024, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.1, mode=train, model=yolov8s

In [ ]:
from ultralytics import YOLO
import torch
import gc
import os
import shutil

# ─────────────────────────────────────────────
# CONFIG — Session 2 (fine-tune from ep25 weights)
# NOTE: last.pt was stripped → can't resume optimizer
# Strategy: low-LR fine-tune for 25 more epochs
# ─────────────────────────────────────────────
MODEL_SIZE     = 's'
EPOCHS         = 25            # 25 more epochs on top of session 1
IMG_SIZE       = 1024
BATCH          = 32
PROJECT        = '/content/runs'
RUN_NAME       = 'yolov8s_session2'
DRIVE_SAVE_DIR = '/content/drive/MyDrive/YOLOv8_weights'

RESUME_WEIGHTS = '/content/drive/MyDrive/YOLOv8_weights/last.pt'

# ─────────────────────────────────────────────
# GPU
# ─────────────────────────────────────────────
torch.cuda.empty_cache()
gc.collect()

print(f"🚀 GPU: {torch.cuda.get_device_name(0)}")
print(f"   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

assert os.path.exists(RESUME_WEIGHTS), (
    f"\n❌ last.pt not found at: {RESUME_WEIGHTS}\n"
    f"   Make sure you added your YOLOv8_weights dataset to this notebook"
)

# ─────────────────────────────────────────────
# Load weights (fine-tune mode, NOT resume)
# ─────────────────────────────────────────────
model = YOLO(RESUME_WEIGHTS)
print(f"✅ Loaded weights from: {RESUME_WEIGHTS}")
print(f"   Parameters: {sum(p.numel() for p in model.model.parameters()):,}")
print(f"   Mode: Fine-tune (low LR, no optimizer resume)")

# ─────────────────────────────────────────────
# Train — Session 2
# Session 1 ended at: 0.882 standard / 0.895 TTA
# Using lower LR to preserve learned weights
# ─────────────────────────────────────────────
print(f"\n🏋️ Session 2: fine-tune {EPOCHS} epochs | imgsz={IMG_SIZE} | batch={BATCH}\n")

results = model.train(
    data=str(yolo_yaml_path),
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH,
    device=0,
    cache='disk',
    workers=4,
    resume=False,              # ← False: stripped weights can't resume

    patience=10,
    save=True,
    save_period=5,
    project=PROJECT,
    name=RUN_NAME,
    exist_ok=True,
    pretrained=True,
    optimizer='AdamW',
    verbose=True,
    seed=42,
    deterministic=False,

    # ── KEY CHANGE: lower LR to fine-tune, not retrain ──
    lr0=0.001,                 # 10× lower than session 1
    lrf=0.01,                  # cosine decays to lr0*lrf = 0.00001
    momentum=0.937,
    weight_decay=0.0005,
    warmup_epochs=1.0,         # shorter warmup — weights already good
    warmup_momentum=0.8,

    cos_lr=True,
    close_mosaic=5,

    box=9.0,
    cls=0.5,
    dfl=1.5,

    amp=True,
    fraction=1.0,
    rect=False,
    single_cls=False,
    profile=False,

    # ── Same augmentation as session 1 ────────────
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    degrees=5.0,
    translate=0.1,
    scale=0.6,
    shear=2.0,
    perspective=0.0001,
    flipud=0.0,
    fliplr=0.5,
    mosaic=1.0,
    mixup=0.1,
    copy_paste=0.1,
)

print("\n✅ Session 2 complete!")

# ─────────────────────────────────────────────
# Save to Google Drive
# IMPORTANT: save last.pt BEFORE it gets stripped
# so Session 3 can resume properly if needed
# ─────────────────────────────────────────────
os.makedirs(DRIVE_SAVE_DIR, exist_ok=True)

best_src = f'{PROJECT}/{RUN_NAME}/weights/best.pt'
last_src = f'{PROJECT}/{RUN_NAME}/weights/last.pt'

shutil.copy(best_src, f'{DRIVE_SAVE_DIR}/best_s2_ep{EPOCHS}.pt')
shutil.copy(last_src, f'{DRIVE_SAVE_DIR}/last_s2_ep{EPOCHS}.pt')
shutil.copy(best_src, f'{DRIVE_SAVE_DIR}/best.pt')   # overwrite with latest best

# ── Save full last.pt for a possible Session 3 ──
# This is the UNSTRIPPED checkpoint with optimizer state
# Use it with resume=True in session 3
shutil.copy(last_src, f'{DRIVE_SAVE_DIR}/last_full_for_resume.pt')
print(f"💾 Saved full checkpoint for session 3 resume: last_full_for_resume.pt")

print(f"\n💾 Saved to Google Drive: {DRIVE_SAVE_DIR}")
print(f"   → best_s2_ep{EPOCHS}.pt          ← best from this session")
print(f"   → best.pt                        ← same, easy access")
print(f"   → last_full_for_resume.pt        ← use this for session 3 resume=True")

# ─────────────────────────────────────────────
# Cleanup
# ─────────────────────────────────────────────
del model
torch.cuda.empty_cache()
gc.collect()

# ─────────────────────────────────────────────
# Final Validation — Standard + TTA
# ─────────────────────────────────────────────
best_model = YOLO(best_src)
print(f"\n📂 Loaded best model for final validation")

# Standard
print("\n🔍 Standard validation...")
val_results = best_model.val(data=str(yolo_yaml_path), imgsz=IMG_SIZE, device=0)
print(f"\n📊 Standard Results:")
print(f"   mAP50:     {val_results.box.map50:.4f}  (session 1: 0.8821)")
print(f"   mAP50-95:  {val_results.box.map:.4f}  (session 1: 0.6511)")
print(f"   Precision: {val_results.box.mp:.4f}  (session 1: 0.8480)")
print(f"   Recall:    {val_results.box.mr:.4f}  (session 1: 0.8090)")

print(f"\n   Per class mAP50:")
names = ['bus', 'car', 'pickup', 'truck', 'van']
s1    = [0.890, 0.878, 0.898, 0.901, 0.845]
for i, (name, s1_score) in enumerate(zip(names, s1)):
    try:
        score = val_results.box.ap50[i]
        print(f"   {name:<8} {score:.3f}  (session 1: {s1_score:.3f})  {'↑' if score > s1_score else '↓'}")
    except:
        pass

# TTA
print("\n🔍 TTA validation (augment=True)...")
tta_results = best_model.val(data=str(yolo_yaml_path), imgsz=IMG_SIZE, device=0, augment=True)
print(f"\n📊 TTA Results:")
print(f"   mAP50:     {tta_results.box.map50:.4f}  (session 1: 0.8952)")
print(f"   mAP50-95:  {tta_results.box.map:.4f}  (session 1: 0.6882)")
print(f"   Precision: {tta_results.box.mp:.4f}  (session 1: 0.8440)")
print(f"   Recall:    {tta_results.box.mr:.4f}  (session 1: 0.8270)")

gain = tta_results.box.map50 - 0.8952
print(f"\n🎯 Gain over session 1 TTA (0.8952): {gain:+.4f}")

if tta_results.box.map50 >= 0.90:
    print("   ✅ TARGET HIT — 90%+ mAP50 achieved!")
    print(f"   📁 Final weights: {DRIVE_SAVE_DIR}/best.pt")
elif tta_results.box.map50 >= 0.895:
    print("   🔶 Very close! Run session 3 using last_full_for_resume.pt with resume=True")
else:
    print("   ⚠️  Model regressed — use best_ep25.pt from session 1 instead")
    print("   💡 Try YOLOv8m with BATCH=16 for a stronger baseline")

🚀 GPU: NVIDIA A100-SXM4-40GB
   VRAM: 42.4 GB
✅ Loaded weights from: /content/drive/MyDrive/YOLOv8_weights/last.pt
   Parameters: 11,137,535
   Mode: Fine-tune (low LR, no optimizer resume)

🏋️ Session 2: fine-tune 25 epochs | imgsz=1024 | batch=32

Ultralytics 8.4.37 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (NVIDIA A100-SXM4-40GB, 40441MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=9.0, cache=disk, cfg=None, classes=None, close_mosaic=5, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.1, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/kaggle/working/data_yolo.yaml, degrees=5.0, deterministic=False, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=25, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1024, int8=False, iou=0.7, keras=False, kobj=1.0

In [ ]:
import ultralytics
from ultralytics import YOLO
import torch
import gc
import os
import shutil
from pathlib import Path

# Clear GPU cache and collect garbage before starting
torch.cuda.empty_cache()
gc.collect()

# Define configuration for this new session (YOLOv10s, 10 epochs)
MODEL_ARCH = 'yolov10s' # User requested 'yolo 10s model'
EPOCHS_NEW_SESSION = 10 # User requested '10 epoc'
IMG_SIZE = 1024 # Reusing from previous sessions
BATCH = 32 # Reusing from previous sessions, adjust if GPU memory issues occur with YOLOv10s
PROJECT = '/content/runs' # Reusing
RUN_NAME_NEW = f'{MODEL_ARCH}_fresh_session' # New run name to distinguish this session
DRIVE_SAVE_DIR = '/content/drive/MyDrive/YOLOv8_weights' # Reusing from previous sessions

# Ensure yolo_yaml_path is available from previous cells
# It should be defined from cell f24368f3 as Path("/kaggle/working/data_yolo.yaml")
# If it's not defined, this line would cause an error. Assuming it is global.
# If `yolo_yaml_path` is not defined globally, you would need to add:
# yolo_yaml_path = Path("/kaggle/working/data_yolo.yaml")

print(f"🚀 GPU: {torch.cuda.get_device_name(0)}")
print(f"   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# Load Model - attempting to load YOLOv10s
try:
    # Ultralytics will attempt to download the model if it's not available locally
    model = YOLO(f'{MODEL_ARCH}.pt')
    print(f"✅ Loaded: {MODEL_ARCH}")
    print(f"   Parameters: {sum(p.numel() for p in model.model.parameters()):,}")
except Exception as e:
    print(f"❌ Error loading {MODEL_ARCH}.pt: {e}")
    print("   This might mean YOLOv10s is not directly supported by this Ultralytics version or not found.")
    print("   Please check Ultralytics documentation for YOLOv10 support or try a different model (e.g., yolov8s). If this error occurs, training will be skipped.")
    model = None # Set model to None if loading fails

if model:
    # Train - 1st session for YOLOv10s
    print(f"\n⚗️ Starting new session for {MODEL_ARCH}: epoch 1 → {EPOCHS_NEW_SESSION} | imgsz={IMG_SIZE} | batch={BATCH}\n")

    results = model.train(
        data=str(yolo_yaml_path), # Using the previously prepared data.yaml
        epochs=EPOCHS_NEW_SESSION,
        imgsz=IMG_SIZE,
        batch=BATCH,
        device=0,
        cache='disk',
        workers=4,
        resume=False,
        patience=min(5, EPOCHS_NEW_SESSION), # Adjust patience for shorter training (max 5 epochs, or less if EPOCHS_NEW_SESSION is smaller)
        save=True,
        save_period=5, # Save every 5 epochs
        project=PROJECT,
        name=RUN_NAME_NEW,
        exist_ok=True,
        pretrained=True, # Use pretrained weights if available
        optimizer='AdamW',
        verbose=True,
        seed=42,
        deterministic=False,

        lr0=0.01,
        lrf=0.01,
        momentum=0.937,
        weight_decay=0.0005,
        warmup_epochs=3.0, # Standard warmup for a new model
        warmup_momentum=0.8,

        cos_lr=True,
        close_mosaic=0, # Keep mosaic on for all epochs for shorter training. If 10 epochs, 5 would disable for half.

        box=9.0,
        cls=0.5,
        dfl=1.5,

        amp=True,
        fraction=1.0,
        rect=False,
        single_cls=False,
        profile=False,

        hsv_h=0.015,
        hsv_s=0.7,
        hsv_v=0.4,
        degrees=5.0,
        translate=0.1,
        scale=0.6,
        shear=2.0,
        perspective=0.0001,
        flipud=0.0,
        fliplr=0.5,
        mosaic=1.0,
        mixup=0.1,
        copy_paste=0.1,
    )

    print(f"\n✅ New training session for {MODEL_ARCH} complete!")

    # Save to Google Drive
    os.makedirs(DRIVE_SAVE_DIR, exist_ok=True)

    best_src = f'{PROJECT}/{RUN_NAME_NEW}/weights/best.pt'
    last_src = f'{PROJECT}/{RUN_NAME_NEW}/weights/last.pt'

    # Save weights with specific names for this YOLOv10s session
    shutil.copy(best_src, f'{DRIVE_SAVE_DIR}/{MODEL_ARCH}_best_ep{EPOCHS_NEW_SESSION}.pt')
    shutil.copy(last_src, f'{DRIVE_SAVE_DIR}/{MODEL_ARCH}_last_ep{EPOCHS_NEW_SESSION}.pt')

    print(f"\n💾 Saved to Google Drive: {DRIVE_SAVE_DIR}")
    print(f"   → {MODEL_ARCH}_best_ep{EPOCHS_NEW_SESSION}.pt")
    print(f"   → {MODEL_ARCH}_last_ep{EPOCHS_NEW_SESSION}.pt")

    # Quick Validation to check performance against the target
    del model
    torch.cuda.empty_cache()
    gc.collect()

    best_model_new = YOLO(best_src)

    print(f"\n🔍 Performing validation for the new {MODEL_ARCH} model...")
    val_results_new = best_model_new.val(data=str(yolo_yaml_path), imgsz=IMG_SIZE, device=0)
    print(f"\n📊 Standard Validation Results:")
    print(f"   mAP50:     {val_results_new.box.map50:.4f}")
    print(f"   mAP50-95:  {val_results_new.box.map:.4f}")

    print(f"\n🔍 Performing TTA (Test Time Augmentation) validation...")
    tta_results_new = best_model_new.val(data=str(yolo_yaml_path), imgsz=IMG_SIZE, device=0, augment=True)
    print(f"📊 TTA Validation Results:")
    print(f"   mAP50:     {tta_results_new.box.map50:.4f}")
    print(f"   mAP50-95:  {tta_results_new.box.map:.4f}")

    target_mAP50 = 0.90
    if tta_results_new.box.map50 >= target_mAP50:
        print(f"\n🎯 TARGET HIT — {target_mAP50 * 100:.0f}% mAP50 achieved with {MODEL_ARCH} in {EPOCHS_NEW_SESSION} epochs!")
        print(f"   Final weights: {DRIVE_SAVE_DIR}/{MODEL_ARCH}_best_ep{EPOCHS_NEW_SESSION}.pt")
    else:
        print(f"\n🎯 Target {target_mAP50 * 100:.0f}% mAP50 not yet reached. Current TTA mAP50: {tta_results_new.box.map50:.4f}")
        print(f"   Consider training for more epochs or fine-tuning this {MODEL_ARCH} model.")
else:
    print("Training was skipped because the model could not be loaded.")

🚀 GPU: NVIDIA A100-SXM4-40GB
   VRAM: 42.4 GB
✅ Loaded: yolov10s
   Parameters: 8,128,272

⚗️ Starting new session for yolov10s: epoch 1 → 10 | imgsz=1024 | batch=32

Ultralytics 8.4.37 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (NVIDIA A100-SXM4-40GB, 40441MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=9.0, cache=disk, cfg=None, classes=None, close_mosaic=0, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.1, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/kaggle/working/data_yolo.yaml, degrees=5.0, deterministic=False, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=10, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1024, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.1, mode=t